In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

# 도구 인스턴스 생성
search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool(website_url='https://www.tripadvisor.co.kr/')

In [10]:
from typing import Type
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

class CalculatorInput(BaseModel):
    expression: str = Field(
        ...,
        description="계산할 수식을 문자열로 입력하세요",
        json_schema_extra={"example": "2+2*3"}  # Pydantic v2 방식 예시
    )


class CalculatorTool(BaseTool):
    name: str = "calculator"
    description: str = "수학 계산을 수행합니다."
    args_schema: Type[BaseModel] = CalculatorInput

    def _run(self, expression: str) -> str:
        try:
            # eval 사용은 위험할 수 있으므로 안전한 eval 또는 parser 권장
            result = eval(expression, {"__builtins__": {}})
            return str(result)
        except Exception as e:
            return f"계산 오류: {e}"


# 도구 인스턴스
calculator_tool = CalculatorTool()

In [11]:
# 결과 저장용 딕셔너리 생성
agent_results = {}

In [13]:
from crewai import Agent, Task, Crew, Process

# 뉴스 분석 Agent
news_agent = Agent(
    role="뉴스 분석가",
    goal="최신 여행 관련 뉴스를 요약하여 전달합니다.",
    backstory="다양한 언론사의 뉴스를 분석하여 최신 여행 정보를 제공합니다.",
    tools=[search_tool, scrape_tool],
    llm="gpt-4.1-mini",
    verbose=True
)

# 여행 전문가 Agent
travel_agent = Agent(
    role="여행 전문가",
    goal="여행 계획을 세우고 일정을 제안합니다.",
    backstory="오랜 여행 기획 경력을 가진 여행 전문가입니다.",
    tools=[search_tool, scrape_tool],
    llm="gpt-4.1-mini",
    verbose=True
)

# 요리 전문가 Agent
recipe_agent = Agent(
    role="요리 전문가",
    goal="여행지의 현지 음식을 찾아 레시피를 제공합니다.",
    backstory="전 세계의 다양한 음식을 연구하고 요리법을 전파하는 요리 연구가입니다.",
    tools=[search_tool, scrape_tool],
    llm="gpt-4.1-mini",
    verbose=True
)

In [14]:
# Task 정의
news_task = Task(
    description="최신 여행 뉴스를 검색하여 한국어로 3가지 주요 뉴스를 요약해 제공합니다.",
    expected_output="최신 여행 뉴스 요약 3가지 (한국어)",
    agent=news_agent,
    callback=lambda output: agent_results.update({"뉴스 요약":str(output)})    # Task가 끈난 후 실행되는 후처리 함수.결과값을 가공하거나 외부 변수에 저장하는데 사용. 
)
travel_task = Task(
    description="앞에서 제공된 최신 여행 뉴스 요약을 바탕으로 한국에서 3일간의 여행 일정을 상세히 작성합니다.",
    expected_output="3일 여행 일정 상세 안내 (한국어)",
    agent=travel_agent,
    # context=[news_task],
    callback=lambda output: agent_results.update({"여행 일정":str(output)}) 
)

recipe_task = Task(
    description="앞서 제시된 여행 일정의 지역 중 한 곳의 대표적인 현지 음식을 선정하고, "
                "웹 검색으로 해당 음식의 레시피를 검색한 후, 한국어로 상세한 레시피를 제공합니다.",
    expected_output="선택된 지역의 대표 현지 음식 레시피 (한국어)",
    agent=recipe_agent,
    # context=[travel_task],
    callback=lambda output: agent_results.update({"레시피":str(output)}) 
)

In [15]:
# Crew 정의 (순차적 프로세스)
crew = Crew(
    agents=[news_agent, travel_agent, recipe_agent],
    tasks=[news_task, travel_task, recipe_task],
    process=Process.sequential,
    verbose=True
)

In [16]:
fianl_result = await crew.kickoff_async()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b398d996-591a-4a40-97b1-9159f5436dd3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 최신 여행 뉴스를 검색하여 한국어로 3가지 주요 뉴스를 요약해 제공합니다.                                  │
│  ID: 035d4449-4d7f-43d4-8a3d-3de8a60fd929                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 뉴스 분석가                                                                                             │
│                                                                                                                 │
│  Task: 최신 여행 뉴스를 검색하여 한국어로 3가지 주요 뉴스를 요약해 제공합니다.                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': '최신 여행 뉴스 2024'}                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': '최신 여행 뉴스 2024', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': '여행신문', 'link': 'https://www.traveltimes.co.kr/', 'snippet': '괌, 유류할증료 10만원 지원 유혹… ·...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 요리 전문가                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  서울 북촌한옥마을을 대표하는 전통 한식 중에서 한정식을 중심으로 한 요리를 선정하여 레시피를 소개합니다. 북촌   │
│  한옥마을 인근 맛집에서 즐길 수 있는 갈비찜, 비빔밥 등 대표적 한식 요리의 레시피를 한국어로 자세히              │
│  설명드리겠습니다.                                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 갈비찜 레시피 (전통 한식 대표 메뉴)                                                                         │
│                                                                                                                 │
│  ### 재료                                                                                                       │
│  - 소 갈비 1kg                                                                                                  │
│  - 무 1/2개 (두툼하게 썰기)                                                                                     │
│  - 당근 1/2개 (두툼하게 썰기)                                                                                   │
│  - 대파 1대                                                                                                     │
│  - 마늘 6쪽 (다진 마늘 혹은 통마늘)                                                                             │
│  - 생강 1 작은술 (다진 생강)                                                                                    │
│  - 간장 1컵                                                                                                     │
│  - 참기름 2큰술                                                                                                 │
│  - 설탕 3큰술                                                                                                   │
│  - 물엿 또는 올리고당 2큰술                                                                                     │
│  - 후춧가루 약간                                                                                                │
│  - 청양고추 1~2개 (선택, 매운맛)                                                                                │
│  - 통깨 약간 (마무리용)                                                                                         │
│                                                                                                                 │
│  ### 조리법                                                                                                     │
│  1. **갈비 손질**                                                                                               │
│     - 찬물에 소 갈비를 2~3시간 정도 담가 핏물을 제거한다. 중간중간 물을 갈아주면 더 좋다.                       │
│     - 끓는 물에 갈비를 넣어 5분 정도 데친 후 찬물에 헹궈 불순물을 제거한다.                                     │
│                                                                                                                 │
│  2. **양념장 만들기**                                                                                           │
│     - 큰 볼에 간장, 설탕, 물엿, 참기름, 다진 마늘, 다진 생강, 후춧가루를 넣고 잘 섞어 양념장을 만든다.          │
│                                                                                                                 │
│  3. **갈비 재우기**                                                                                             │
│     - 손질한 갈비를 양념장에 넣고 최소 1시간 이

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 앞서 제시된 여행 일정의 지역 중 한 곳의 대표적인 현지 음식을 선정하고, 웹 검색으로 해당 음식의 레시피를  │
│  검색한 후, 한국어로 상세한 레시피를 제공합니다.                                                                │
│  Agent: 요리 전문가                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b398d996-591a-4a40-97b1-9159f5436dd3                                                                       │
│  Final Output: 서울 북촌한옥마을을 대표하는 전통 한식 중에서 한정식을 중심으로 한 요리를 선정하여 레시피를      │
│  소개합니다. 북촌 한옥마을 인근 맛집에서 즐길 수 있는 갈비찜, 비빔밥 등 대표적 한식 요리의 레시피를 한국어로    │
│  자세히 설명드리겠습니다.                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 갈비찜 레시피 (전통 한식 대표 메뉴)                                                                         │
│                                                                                                                 │
│  ### 재료                                                                                                       │
│  - 소 갈비 1kg                                                                                                  │
│  - 무 1/2개 (두툼하게 썰기)                                                                                     │
│  - 당근 1/2개 (두툼하게 썰기)                                                                                   │
│  - 대파 1대                                                                                                     │
│  - 마늘 6쪽 (다진 마늘 혹은 통마늘)                                                                             │
│  - 생강 1 작은술 (다진 생강)                                                                                    │
│  - 간장 1컵                                                                                                     │
│  - 참기름 2큰술                                                                                                 │
│  - 설탕 3큰술                                                                                                   │
│  - 물엿 또는 올리고당 2큰술                                                                                     │
│  - 후춧가루 약간                                                                                                │
│  - 청양고추 1~2개 (선택, 매운맛)                                                                                │
│  - 통깨 약간 (마무리용)                                                                                         │
│                                                                                                                 │
│  ### 조리법                                                                                                     │
│  1. **갈비 손질**                                                                                               │
│     - 찬물에 소 갈비를 2~3시간 정도 담가 핏물을 제거한다. 중간중간 물을 갈아주면 더 좋다.                       │
│     - 끓는 물에 갈비를 넣어 5분 정도 데친 후 찬물에 헹궈 불순물을 제거한다.                                     │
│                                                                                                                 │
│  2. **양념장 만들기**                                                                                           │
│     - 큰 볼에 간장, 설탕, 물엿, 참기름, 다진 마늘, 다진 생강, 후춧가루를 넣고 잘 섞어 양념장을 만든다.          │
│                                                                                                                 │
│  3. **갈비 재우기**                                                                                             │
│     - 손질한 갈비를 양념장에 넣고 최소

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
print("\n--- 최종 결과 출력 ---\n")

for key, value in agent_results.items():
    print(f"\n✅ {key}\n{value}\n")


--- 최종 결과 출력 ---


✅ 뉴스 요약
2024년 최신 여행 뉴스 3가지 주요 소식 요약입니다.

1. 2024년 여행 트렌드: 테마 로컬 여행과 숨겨진 명소 인기
   - 2024년 여름 국내 여행에서는 '테마 관광'과 '숨겨진 명소', '촌캉스'(시골에서의 휴가) 여행이 대세로 떠올랐습니다. 한국관광공사에 따르면 가족, 친구 등 전통적인 여행 단위를 넘어 반려동물과 함께하는 여행, 나홀로 여행, 시니어 관광 등 다양성이 확대되고 있습니다. 즉흥적인 모험을 추구하는 '깜짝 여행'과 개인화, 경험 중심의 여행 수요도 증가하고 있습니다.

2. 한국인이 가장 많이 찾는 2024년 인기 해외 여행지
   - 한국인이 가장 많이 찾은 해외 여행지는 여전히 일본으로, 2023년에 전년 대비 7.1% 증가한 약 945만 명이 방문했습니다. 그 다음으로는 베트남과 태국이 상위권을 차지했으며, 베트남은 가성비와 문화 매력으로 큰 인기를 끌고 있습니다. 미국, 인도네시아 발리 등도 2024년 한국인 인기 여행지로 꼽히고 있습니다.

3. 2024년 해외여행 전망과 주요 트렌드
   - 2024년에도 해외여행 열기는 이어질 전망이며, 스카이스캐너는 7대 여행 트렌드로 엔터투어먼트, 성지 투어, 맛성비 미식가, 꿀잠 여행, 아날로그 여행, 기념 여행, 스몰 럭셔리 여행을 제시했습니다. 또한, 디지털 기술과 가상현실, 증강현실을 활용한 경험형 여행도 확산될 것으로 보이며, 환경보호 등을 고려하는 책임 여행이 중요하게 다뤄지고 있습니다.

요약하자면, 2024년 여행은 개인의 취향과 경험을 중시하며 국내외 모두에서 새로운 여행지와 테마를 찾아 나서는 경향이 강해지고 있습니다. 한국인은 일본을 비롯한 아시아권 국가에 대한 여행 수요가 높고, 해외여행 트렌드 역시 보다 다양하고 지속 가능한 방향으로 진화하고 있습니다.


✅ 여행 일정
2024년 최신 여행 뉴스와 트렌드를 반영한 한국 내 3일간의 여행 일정을 다음과 같이 상세히 제안드립니다. 이번 일정은 ‘테마 로컬 여행’